In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# 06 — Validation, Regularisation, Threshold Tuning & Business Interpretation

**Course:** Data Analysis, AI and Machine Learning in Finance — Ferran Carrascosa
**Session 3 — closing notebook**

---

### What this notebook covers

| Section | Topic |
|---------|-------|
| 0 | Why validation matters |
| 1 | Rebuild the best candidate models |
| 2 | Train vs test behaviour and overfitting signs |
| 3 | Class imbalance awareness |
| 4 | **Threshold tuning** — from default to business-aligned |
| 5 | Regularisation intuition |
| 6 | Cross-validation for robustness |
| 7 | Lightweight hyperparameter tuning |
| 8 | Interpretation beyond raw score |
| 9 | From model result to business recommendation |
| 10 | Production awareness — monitoring, drift, retraining |
| 11 | Self-practice tasks |

**Dataset:** `dataset_C_bank_attrition_core.csv` (same as Notebooks 04 / 05 / 05b)
**Target:** `attrition_flag` (1 = churned, 0 = stayed)

> *"A model is not useful just because it had one good day."*

---

## Section 0 — Why validation matters

In Notebooks 05 and 05b we trained several models and compared their metrics on a
single train/test split. The results looked promising:

| Model | AUC-ROC | Recall |
|-------|---------|--------|
| Logistic Regression | 0.871 | 0.447 |
| Decision Tree (depth=4) | 0.860 | 0.511 |
| Random Forest | 0.910 | 0.523 |
| XGBoost | 0.911 | 0.624 |

But a single evaluation is **not enough** to trust a model. Why?

1. **Stability** — did we just get lucky with the random split?
2. **Trust** — would the model behave similarly on tomorrow's customers?
3. **Governance** — regulators expect proof of robustness, not a single number.
4. **Production risk** — a model that works on one sample but fails on another is
   worse than no model at all.

This notebook adds **validation discipline**, **threshold awareness** and
**business interpretation** — the layer that turns a score into a decision.

---

## Section 1 — Rebuild the best candidate models

We rebuild the modelling table and re-train the three key models from the earlier
notebooks so everything is self-contained and reproducible.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

# ── Load dataset ──────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / "dataset_C_bank_attrition_core.csv")

# ── One-hot encoding (same as NB04) ──────────────────────────────────
df_encoded = pd.get_dummies(df, columns=["geography", "gender"], drop_first=True)

target_col = "attrition_flag"
id_col     = "customer_id"

y = df_encoded[target_col]
X = df_encoded.drop(columns=[target_col, id_col])

print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Features: {X.shape[1]}  |  Target: {target_col}")
print(f"Churn rate: {y.mean():.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay,
)

# ── Train / test split (identical to NB04) ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y,
)

# ── Scaled version for logistic regression ───────────────────────────
scaler = StandardScaler()
X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_sc = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(f"Training: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Churn rate — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

In [ ]:
# ── 1. Logistic Regression baseline ──────────────────────────────────
logreg = LogisticRegression(max_iter=1000, random_state=SEED)
logreg.fit(X_train_sc, y_train)

# ── 2. Random Forest ─────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)

# ── 3. XGBoost ───────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric="logloss",
)
xgb.fit(X_train, y_train)

print("Three candidate models trained: LogReg, Random Forest, XGBoost.")

> We intentionally rebuild all three models here so that this notebook can be
> run independently without relying on earlier notebook state.

---

## Section 2 — Train vs test behaviour and overfitting signs

A model that performs brilliantly on training data but poorly on test data has
**memorised** rather than **learned**. This is called *overfitting*.

The simplest diagnostic: compare common metrics on both sets and look at the
**gap**. A small gap is healthy; a large gap is a warning sign.

In [ ]:
def eval_set(model, X_tr, X_te, y_tr, y_te, name, scale=False):
    """Compute train and test metrics for a given model."""
    Xtr = X_tr if not scale else X_train_sc
    Xte = X_te if not scale else X_test_sc
    metrics = {}
    for label, Xs, ys in [("Train", Xtr, y_tr), ("Test", Xte, y_te)]:
        yp = model.predict(Xs)
        ypr = model.predict_proba(Xs)[:, 1]
        metrics[label] = {
            "Accuracy": accuracy_score(ys, yp),
            "Precision": precision_score(ys, yp),
            "Recall": recall_score(ys, yp),
            "F1": f1_score(ys, yp),
            "AUC-ROC": roc_auc_score(ys, ypr),
        }
    row = {"Model": name}
    for m in ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]:
        row[f"{m} (Train)"] = metrics["Train"][m]
        row[f"{m} (Test)"]  = metrics["Test"][m]
        row[f"{m} Gap"]     = metrics["Train"][m] - metrics["Test"][m]
    return row

rows = [
    eval_set(logreg, X_train, X_test, y_train, y_test, "Logistic Regression", scale=True),
    eval_set(rf,     X_train, X_test, y_train, y_test, "Random Forest"),
    eval_set(xgb,    X_train, X_test, y_train, y_test, "XGBoost"),
]
gap_df = pd.DataFrame(rows).set_index("Model")

# Show the key AUC and Recall gaps
summary_cols = ["AUC-ROC (Train)", "AUC-ROC (Test)", "AUC-ROC Gap",
                "Recall (Train)", "Recall (Test)", "Recall Gap"]
print("Train vs Test Comparison")
print("=" * 60)
print(gap_df[summary_cols].round(3).to_string())

### What to look for

| Signal | Interpretation |
|--------|---------------|
| **AUC Gap < 0.02** | Healthy — the model generalises well |
| **AUC Gap 0.02 – 0.05** | Moderate — keep an eye on it |
| **AUC Gap > 0.05** | Warning — the model may be overfitting |
| **Recall Gap > 0.10** | The model catches churners in training but misses them in test — unreliable |

> **Reflection:** logistic regression typically has the smallest gap (low
> complexity = low overfitting risk). XGBoost and Random Forest are more
> powerful but must be checked for overfitting — *power and discipline go
> hand in hand*.

---

## Section 3 — Class imbalance awareness

Our target has approximately **80 % stayed / 20 % churned**. This ratio matters
because a lazy model that predicts *"stayed"* for every customer would still be
80 % accurate — yet it would be completely useless for identifying churn risk.

This is why we need metrics beyond accuracy.

In [ ]:
# Class distribution
counts = y_test.value_counts()
print("Test set class distribution")
print("-" * 35)
for val, cnt in counts.items():
    label = "Churned" if val == 1 else "Stayed"
    print(f"  {label} ({val}): {cnt:>5}  ({cnt / len(y_test):.1%})")
print()

# Illustrate the "always predict stayed" baseline
naive_preds = pd.Series(0, index=y_test.index)
print(f"Naive 'always stayed' baseline:")
print(f"  Accuracy = {accuracy_score(y_test, naive_preds):.3f}")
print(f"  Recall   = {recall_score(y_test, naive_preds):.3f}  ← catches ZERO churners")
print(f"  F1       = {f1_score(y_test, naive_preds):.3f}")
print()
print("→ Accuracy alone can be dangerously misleading.")

### Metric refresher for imbalanced targets

| Metric | Meaning in our context | Why it matters |
|--------|----------------------|----------------|
| **Precision** | Of customers we flag as churners, how many actually churn? | Controls **false positives** — wasted retention spend |
| **Recall** | Of all real churners, how many do we catch? | Controls **false negatives** — lost customers we failed to help |
| **F1** | Harmonic mean of precision and recall | Balances both concerns |
| **AUC-ROC** | Overall ranking ability across all thresholds | Threshold-independent quality measure |

> In a **customer retention** scenario, missing a real churner (low recall) often
> costs more than wrongly contacting a happy customer (low precision). But the
> right balance depends on the business context and the cost of each action.

> **From confusion matrix to financial cost**  
> In finance, a confusion matrix is not just a technical summary — it is a cost map.  
> **False positives** may mean unnecessary retention calls or review costs.  
> **False negatives** may mean missed churners, undetected fraud, or lost revenue.  
>  
> **Precision** tells us how much of our flagged population is truly relevant: it behaves like an **efficiency / risk-control** measure.  
> **Recall** tells us how much of the real positive class we actually capture: it behaves like a **business capture** measure.  
> **F1 score** gives a single compromise when we need both to be reasonable at the same time.


---

## Section 4 — Threshold tuning: from default to business-aligned

### The hidden assumption

Every classification model outputs a **probability** (e.g., 0.37 or 0.72). To
make a decision — *"flag this customer as churn risk or not"* — we compare that
probability to a **threshold**.

By default, scikit-learn and XGBoost use **threshold = 0.5**:

- probability ≥ 0.5 → predict *churned*
- probability < 0.5 → predict *stayed*

But **0.5 is not a law of nature**. It is just a convenient default. In practice,
the right threshold depends on the **relative cost** of two types of mistakes:

| Mistake | Name | Business impact |
|---------|------|-----------------|
| Predict "stayed" but customer churns | **False Negative** | Lost revenue — we missed a customer we could have retained |
| Predict "churned" but customer stays | **False Positive** | Wasted retention budget — we spent resources on a loyal customer |

> **Key insight:** lowering the threshold catches more churners (↑ recall) but
> also sends more false alarms (↓ precision). Raising the threshold does the
> opposite. There is no free lunch — only a **business trade-off**.

In [ ]:
# ── Threshold sweep on XGBoost (our strongest model) ─────────────────
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.10, 0.91, 0.05)
threshold_rows = []
for t in thresholds:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    threshold_rows.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(y_test, y_pred_t, zero_division=0),
        "Recall":    recall_score(y_test, y_pred_t, zero_division=0),
        "F1":        f1_score(y_test, y_pred_t, zero_division=0),
        "Flagged":   y_pred_t.sum(),
    })

thresh_df = pd.DataFrame(threshold_rows).round(3)
print(thresh_df.to_string(index=False))

In [ ]:
# ── Precision–Recall trade-off chart ──────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(thresh_df["Threshold"], thresh_df["Precision"],
         marker="o", label="Precision", color="#5b9bd5")
ax1.plot(thresh_df["Threshold"], thresh_df["Recall"],
         marker="s", label="Recall", color="#ed7d31")
ax1.plot(thresh_df["Threshold"], thresh_df["F1"],
         marker="^", label="F1", linestyle="--", color="#70ad47")

ax1.axvline(0.50, color="grey", linestyle=":", label="Default threshold (0.50)")
ax1.axvline(0.35, color="red",  linestyle=":", alpha=0.7, label="Business threshold (0.35)")
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Score")
ax1.set_title("Precision, Recall and F1 vs Classification Threshold (XGBoost)")
ax1.legend(loc="center left")
ax1.set_ylim(0, 1.05)
ax1.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the chart

- At **threshold = 0.50** (default): precision is relatively high but recall is
  lower — we miss many real churners.
- At **threshold = 0.35** (illustrative business choice): recall jumps noticeably
  — we catch more churners, but precision drops — more false alarms.
  *Note: 0.35 is not a universal rule. The right threshold depends entirely on
  the cost structure of each specific business context.*
- The **F1 curve** peaks somewhere between 0.30 and 0.45, suggesting a good
  balance point.

> **Business question:** *"Is it more expensive to miss a churner or to contact a
> loyal customer unnecessarily?"*
>
> If retention outreach costs €50 per customer and losing a churner costs €500 in
> lifetime value, then false negatives are **10× more expensive** than false
> positives. In that case, a lower threshold (higher recall) makes business
> sense.

In [ ]:
# ── Compare two concrete thresholds ──────────────────────────────────
def threshold_report(y_true, y_prob, t, label):
    y_pred = (y_prob >= t).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Threshold": label,
        "True Pos (caught churners)":  tp,
        "False Neg (missed churners)": fn,
        "False Pos (false alarms)":    fp,
        "True Neg":                    tn,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
    }

t_default  = threshold_report(y_test, y_prob_xgb, 0.50, "0.50 (default)")
t_business = threshold_report(y_test, y_prob_xgb, 0.35, "0.35 (business)")

compare_df = pd.DataFrame([t_default, t_business]).set_index("Threshold")
print(compare_df.to_string())

### The cost perspective

Suppose:
- Retention action cost = **€50** per flagged customer
- Cost of losing a churner = **€500** (lost revenue over 2 years)

| Threshold | False Alarms (FP × €50) | Missed Churners (FN × €500) | Total estimated cost |
|-----------|------------------------|----------------------------|---------------------|

*(Fill in the numbers from the output above as a practice exercise.)*

> **Key takeaway:** threshold tuning is not a statistical nicety — it is a
> **business decision** that directly affects how much the model costs or saves.
> The "best" threshold is the one that minimises total business cost, not the one
> that maximises a single metric.

---

## Section 5 — Confusion matrix at the chosen threshold

A confusion matrix shows the four possible outcomes for every prediction. It is
the most concrete way to see what a model actually does.

In [ ]:
# ── Confusion matrices side by side ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, t, title in [
    (axes[0], 0.50, "Threshold = 0.50 (default)"),
    (axes[1], 0.35, "Threshold = 0.35 (business)"),
]:
    y_pred_t = (y_prob_xgb >= t).astype(int)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred_t,
        display_labels=["Stayed", "Churned"],
        cmap="Blues", ax=ax,
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

### How to read the confusion matrix

|  | Predicted Stayed | Predicted Churned |
|--|-----------------|------------------|
| **Actually Stayed** | True Negative (correct) | False Positive (false alarm) |
| **Actually Churned** | False Negative (missed!) | True Positive (caught) |

- **Top-right** = false alarms → retention budget wasted
- **Bottom-left** = missed churners → revenue lost
- Moving from threshold 0.50 to 0.35 shifts cases from bottom-left to top-right:
  we catch more churners but accept more false alarms.

> The confusion matrix makes the precision–recall trade-off **visible and
> concrete**.

---

## Section 6 — Regularisation intuition

### What is regularisation?

Regularisation is a technique that **penalises model complexity** to prevent
overfitting. Think of it as adding a "simplicity preference" to the model:

> *"If two models perform equally well on training data, prefer the simpler one."*

In everyday language:
- A model without regularisation tries to fit **every detail** in the training
  data, including noise and one-off patterns.
- A regularised model accepts a slightly worse training fit in exchange for
  **better generalisation** to new data.

### How it works in practice

| Model family | Regularisation mechanism | Intuitive effect |
|-------------|------------------------|-----------------|
| **Logistic Regression** | Penalty on large coefficients (L1 or L2) | Prevents any single feature from dominating the prediction |
| **XGBoost / boosted trees** | `max_depth`, `learning_rate`, `reg_alpha`, `reg_lambda` | Controls how deep and aggressive each tree can be |
| **Random Forest** | `max_depth`, `min_samples_leaf` | Prevents individual trees from growing to extreme depth |

> **Analogy:** regularisation is like a study plan that says *"don't memorise the
> textbook word by word — understand the key concepts so you can answer unseen
> exam questions."*

In [ ]:
# ── Regularisation demo: logistic regression with different C values ──
from sklearn.pipeline import make_pipeline

reg_results = []
for C_val in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C_val, max_iter=1000, random_state=SEED)
    )
    model.fit(X_train, y_train)
    y_tr_pred = model.predict(X_train)
    y_te_pred = model.predict(X_test)
    y_te_prob = model.predict_proba(X_test)[:, 1]
    reg_results.append({
        "C": C_val,
        "Train Acc": accuracy_score(y_train, y_tr_pred),
        "Test Acc":  accuracy_score(y_test, y_te_pred),
        "Test AUC":  roc_auc_score(y_test, y_te_prob),
        "Test Recall": recall_score(y_test, y_te_pred),
    })

reg_df = pd.DataFrame(reg_results)
print("Logistic Regression — effect of regularisation strength (C)")
print("(Lower C = stronger regularisation)")
print()
print(reg_df.round(4).to_string(index=False))

### Reading the results

- **C = 0.001** (very strong regularisation): the model is overly simple — it
  under-fits and test performance suffers.
- **C = 1.0** (scikit-learn default): a reasonable balance.
- **C = 100** (very weak regularisation): the model has almost no penalty — it
  fits training data closely and may overfit.

> The ideal C sits in a *"Goldilocks zone"* — not too simple, not too complex.
> Finding it is part of **hyperparameter tuning** (next section).

---

## Section 7 — Cross-validation for robustness

A single train/test split gives us one estimate of performance. But how **stable**
is that estimate? Cross-validation answers this by repeating the evaluation on
multiple different splits.

### How 5-fold cross-validation works

1. Split the training data into 5 equal parts ("folds").
2. Train on 4 folds, evaluate on the 1 held-out fold.
3. Repeat 5 times, each time holding out a different fold.
4. Report the **mean** and **standard deviation** of each metric.

> A model with high mean AUC but large std is **unstable** — it might perform
> well on some splits and poorly on others. We want **high mean + low std**.

In [ ]:
from sklearn.model_selection import cross_val_score

# Cross-validate each model on AUC-ROC
cv_results = {}

# Logistic Regression (needs scaling — use pipeline)
lr_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=SEED)
)
scores_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["Logistic Regression"] = scores_lr

# Random Forest
scores_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["Random Forest"] = scores_rf

# XGBoost
scores_xgb = cross_val_score(xgb, X_train, y_train, cv=5, scoring="roc_auc")
cv_results["XGBoost"] = scores_xgb

print("5-Fold Cross-Validation — AUC-ROC")
print("=" * 55)
for name, scores in cv_results.items():
    print(f"  {name:25s}  mean={scores.mean():.3f}  std={scores.std():.3f}  "
          f"folds={[round(float(s), 3) for s in scores]}")

### Interpreting the results

| What to check | Good sign | Warning sign |
|---------------|-----------|-------------|
| **Mean AUC** | > 0.85 | < 0.75 |
| **Std of AUC** | < 0.02 | > 0.04 |
| **Consistency** | All folds within ±0.02 of mean | One fold much lower than the rest |

> If XGBoost has the highest mean AUC **and** a low standard deviation, we have
> stronger evidence that its performance is real — not just an artefact of one
> lucky split.

---

## Section 8 — Lightweight hyperparameter tuning

Hyperparameters are settings we choose **before** training — like `max_depth`,
`learning_rate` or `n_estimators`. The model cannot learn these from data; we
must choose them ourselves.

In a classroom setting, we keep this simple:
- Try a small grid of sensible values.
- Use cross-validation to compare them.
- Pick the best combination.

We focus on XGBoost since it was our strongest candidate.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Small, classroom-friendly parameter grid
param_grid = {
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.05, 0.10],
    "n_estimators":  [100, 200],
}

grid_search = GridSearchCV(
    XGBClassifier(
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, eval_metric="logloss",
    ),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True,
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC-ROC: {grid_search.best_score_:.4f}")

In [ ]:
# Show all grid results sorted by AUC
grid_df = pd.DataFrame(grid_search.cv_results_)
grid_df = grid_df.sort_values("rank_test_score")
cols = ["param_max_depth", "param_learning_rate", "param_n_estimators",
        "mean_test_score", "std_test_score", "rank_test_score"]
print(grid_df[cols].head(12).to_string(index=False))

### Observations

- The differences between configurations are often **small** — this is typical
  in practice.
- A simpler configuration that performs nearly as well as the best may be
  preferred for **interpretability** and **stability**.
- Hyperparameter tuning is useful but should not be over-engineered: a 0.002
  AUC improvement is rarely worth significant added complexity.

> **Rule of thumb:** if a simpler setting gives AUC within 0.005 of the best,
> prefer the simpler one.

---

## Section 9 — Evaluate the tuned model with business threshold

Let us now evaluate the grid-search winner on the **test set** at both the
default and business thresholds.

In [ ]:
# ── Best model from grid search ───────────────────────────────────────
best_xgb = grid_search.best_estimator_
y_prob_best = best_xgb.predict_proba(X_test)[:, 1]

# Metrics at two thresholds
for t, label in [(0.50, "Default (0.50)"), (0.35, "Business (0.35)")]:
    y_pred_t = (y_prob_best >= t).astype(int)
    print(f"\nThreshold: {label}")
    print(f"  Accuracy:  {accuracy_score(y_test, y_pred_t):.3f}")
    print(f"  Precision: {precision_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  Recall:    {recall_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  F1:        {f1_score(y_test, y_pred_t, zero_division=0):.3f}")
    print(f"  AUC-ROC:   {roc_auc_score(y_test, y_prob_best):.3f}")

# Confusion matrix for business threshold
print("\nConfusion Matrix at threshold 0.35:")
y_pred_biz = (y_prob_best >= 0.35).astype(int)
cm = confusion_matrix(y_test, y_pred_biz)
print(pd.DataFrame(cm,
    index=["Actually Stayed", "Actually Churned"],
    columns=["Predicted Stayed", "Predicted Churned"],
))

> Compare these numbers with the original XGBoost (NB05b). Tuning and
> threshold adjustment together can improve **practical usefulness** even if
> the AUC barely changes — because we are now optimising for the **right
> business metric**, not just accuracy.

> **A note on AUC and calibration:** a strong AUC-ROC means the model
> *ranks* churners above non-churners effectively, but it does **not**
> guarantee that the predicted probabilities are well-calibrated (i.e.,
> that a predicted 0.35 truly corresponds to a 35 % churn chance).
> In production, probability calibration (e.g., Platt scaling or isotonic
> regression) may be needed before using raw scores for threshold-based
> decisions.

---

## Section 10 — Interpretation beyond raw score

Feature importance tells us which variables the model relies on most, but
**importance is not causality**.

- A feature that ranks high may be a proxy for something else.
- Importance varies by method (gain, permutation, SHAP).
- Business context must validate what the model suggests.

In [ ]:
# ── Feature importance from the tuned XGBoost ────────────────────────
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_xgb.feature_importances_,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi["feature"], fi["importance"], color="#70ad47", edgecolor="white")
ax.set_xlabel("Importance (gain)")
ax.set_title("XGBoost (Tuned) — Feature Importance")
plt.tight_layout()
plt.show()

### Interpretation checklist

Before trusting feature importance, ask:

1. **Does the ranking make business sense?** If `age` and `is_active_member`
   rank high, that aligns with banking intuition about churn drivers.
2. **Is the ranking stable?** Run the model with a different random seed or
   slightly different parameters. If the ranking changes dramatically, it is
   not robust.
3. **Does the feature cause the outcome?** Correlation ≠ causation. A feature
   may be a proxy or a confound.

> **For stakeholders:** present the top 3–5 features with plain-language
> explanations, always noting that *"these are the features the model found most
> useful, not necessarily the root causes of churn."*

---

## Section 11 — From model result to business recommendation

### Executive recommendation

| Element | Details |
|---------|--------|
| **Recommendation** | Deploy the tuned XGBoost model as the primary churn-risk scoring engine, using a classification threshold of **0.35** (illustrative — adjust to actual cost structure). |
| **Expected benefit** | At threshold 0.35 the model identifies approximately **70 %** of at-risk customers (recall ≈ 0.71) with AUC-ROC ≈ 0.91, enabling proactive retention outreach before churn occurs. |
| **Main risk** | The model has been validated on a single train/test split of a classroom-adapted dataset with no temporal hold-out. Performance on truly future data is unverified; probability calibration has not been tested. |
| **Next steps** | 1) Validate on a recent time period (temporal hold-out). 2) Pilot with a small customer segment to measure real-world lift. 3) Calibrate predicted probabilities. 4) Establish a monitoring dashboard for data drift and model degradation. |

> **Exercise:** write your own four-sentence version of this recommendation
> before reading on. Focus on: what to do, why, what could go wrong, and what
> to check first.

---

## Section 12 — Production awareness: monitoring, drift and retraining

Deploying a model is not the end — it is the beginning of a new responsibility.

### Things that can go wrong after deployment

| Risk | Description | Example |
|------|------------|---------|
| **Data drift** | The input data distribution changes over time | A new product launch changes customer behaviour patterns |
| **Concept drift** | The relationship between features and the target changes | Churn drivers shift from price sensitivity to service quality |
| **Model degradation** | Performance drops gradually without a single breaking event | The model still runs, but recall falls from 0.75 to 0.50 over 6 months |
| **Data quality issues** | Missing values, wrong formats, or stale data | A data pipeline breaks and `digital_usage_proxy` becomes all zeros |

### What responsible model management looks like

1. **Monitor key metrics** — track precision, recall and AUC on live predictions.
   Set up alerts when performance drops below a threshold (e.g., AUC < 0.80).
2. **Retrain periodically** — refresh the model on recent data (e.g., quarterly).
3. **Document assumptions** — record which features are used, what thresholds
   were chosen, and what business context justified the decisions.
4. **Keep a challenger model** — maintain a simple baseline (e.g., logistic
   regression) alongside the production model. If the complex model degrades
   faster, the simple one can be a fallback.
5. **Governance** — ensure compliance with internal policies and external
   regulations (e.g., GDPR, fair lending laws).

> *"A model is part of a decision system, not a magic answer."*

---

### Limitations of this analysis

| Limitation | Impact |
|-----------|--------|
| Single dataset (no temporal split) | We cannot verify how the model performs on truly future data |
| No feature interaction analysis | Some feature combinations may matter more than individual features |
| No cost-sensitive learning | We chose the threshold post-hoc; a cost-sensitive objective could be more principled |
| Classroom-adapted labels | The target variable uses derived teaching features; no temporal deployment validation has been performed |
| No A/B testing | The true lift of the model can only be measured through a controlled experiment |

These limitations are normal at the **prototype stage**. The goal is awareness,
not perfection.

---

### Self-practice tasks

1. **Threshold comparison:** pick a third threshold (e.g., 0.25 or 0.45) and
   compute precision, recall and F1. Explain in 2–3 sentences when this
   threshold would make business sense.
2. **Accuracy critique:** write a short paragraph explaining why accuracy alone
   is insufficient for this business problem.
3. **Simpler model test:** re-run the XGBoost grid search with `max_depth` ∈
   {2, 3} only. Does the simpler model lose much AUC? Would you still
   recommend it? Justify.
4. **Stakeholder memo:** write a four-sentence "go / no-go / pilot"
   recommendation for a bank manager who has never seen a confusion matrix.

---

**End of Notebook 06.**

This closes the core modelling cycle of Topic 2. In the next topic we
explore more advanced techniques — but the validation discipline, threshold
awareness and business framing introduced here apply to **every model you
will ever build**.